# Member A -- Learning Rate Experiments

Baseline config comes from `shared_train.py`. Only `learning_rate` varies
across these 10 runs -- everything else must stay at the shared baseline
so results are comparable across the whole group.

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py "autorom[accept-rom-license]" pandas
!AutoROM --accept-license

## Get shared_train.py

Clone the group repo (or upload `shared_train.py` to this Colab session)
so the import below works.

In [ ]:
import os

REPO_DIR = "/content/deep-q-learning-formative3"
if os.path.exists(REPO_DIR):
    # reset --hard (not pull) so this always matches the remote exactly,
    # even if history upstream was rewritten (force-pushed) since your
    # last clone -- a plain pull can't reconcile that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/Mikekimm/deep-q-learning-formative3.git {REPO_DIR}
%cd {REPO_DIR}

from shared_train import BASELINE_CONFIG, GAME_ID, TOTAL_TIMESTEPS, SEED, train_one_run
print(GAME_ID, TOTAL_TIMESTEPS, SEED)
print(BASELINE_CONFIG)

## Smoke test -- run this ONE cell before anything below

This runs the exact same pipeline as a real experiment, but for only
2,000 timesteps (~1-2 minutes) instead of 200,000 (~hours). It exists
to catch install/setup errors early. Do NOT use "Run All" on this
notebook -- run this cell by itself first, confirm it prints a result
with no errors, THEN move on to the real experiment cells below.

In [ ]:
import shared_train

shared_train.TOTAL_TIMESTEPS = 2000  # temporary override for this test only

smoke_row = shared_train.train_one_run(overrides={}, run_name="smoketest", member="test")
print(smoke_row)

In [ ]:
# 10 learning_rate values spanning the useful range; everything else
# stays at BASELINE_CONFIG.
lr_values = [1e-2, 5e-3, 1e-3, 7e-4, 5e-4, 3e-4, 1e-4, 7e-5, 5e-5, 1e-5]

## Optional: watch reward curves live while training runs

Run this once before starting the loop below to open an inline
TensorBoard dashboard -- it updates live as each run trains, so you can
watch reward trends without waiting for a run to finish. (This shows the
reward *graph*, not the game itself -- see the README for why we don't
render pixels during training.)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tb_logs

In [ ]:
results = []
for i, lr in enumerate(lr_values, start=1):
    row = train_one_run(
        overrides={"learning_rate": lr},
        run_name=f"memberA_lr_{i:02d}",
        member="MemberA",
    )
    results.append(row)

## Notes

After each run, record what you observed (reward trend, stability, did
it diverge or plateau early?) -- this becomes the "noted behavior" column
in the report table. Explain *why* you think the effect happened, not
just what happened.